In [1249]:
# library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sklearn
from fifacodes import Members

In [1250]:
# load kaggle data
kaggle = pd.read_csv('data/train.csv')

In [1251]:
print(kaggle.shape)

kaggle.columns

(192, 24)


Index(['version', 'team', 'continent', 'is_host', 'goals_scored_last_4y',
       'goals_received_last_4y', 'wins_last_4y', 'losses_last_4y',
       'draws_last_4y', 'world_cup_titles_before',
       'squad_total_market_value_eur', 'fifa_rank_pre_tournament',
       'fifa_points_pre_tournament', 'squad_avg_age',
       'world_cup_participations_before', 'groups_passed_before',
       'round16_before', 'quarterfinals_before', 'semifinals_before',
       'finals_before', 'winner', 'finalist', 'semi_finalist',
       'quarter_finalist'],
      dtype='object')

In [1252]:
kaggle.iloc[:,0:12].describe()

,version,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,squad_total_market_value_eur,fifa_rank_pre_tournament
count,192.00000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,1.600000e+02,192.000000
mean,2012.00000,0.036458,84.583333,44.572917,26.046875,11.052083,11.802083,0.427083,3.433053e+08,23.541667
std,6.84916,0.187918,23.731776,13.608009,6.912534,4.745612,3.496055,1.085312,3.489032e+08,18.214497
min,2002.00000,0.000000,28.000000,19.000000,8.000000,2.000000,4.000000,0.000000,6.270000e+06,1.000000
25%,2006.00000,0.000000,66.000000,35.000000,21.000000,8.000000,10.000000,0.000000,9.385750e+07,9.000000
50%,2012.00000,0.000000,83.500000,42.500000,25.000000,10.500000,12.000000,0.000000,2.179400e+08,20.000000
75%,2018.00000,0.000000,102.000000,53.250000,31.000000,14.000000,14.000000,0.000000,4.129750e+08,34.000000
max,2022.00000,1.000000,156.000000,95.000000,48.000000,30.000000,23.000000,5.000000,1.620000e+09,105.000000


In [1253]:
kaggle.iloc[:,12:].describe()

,fifa_points_pre_tournament,squad_avg_age,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
count,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000,192.000000
mean,954.165469,27.249635,5.625000,3.572917,2.244792,2.223958,1.348958,0.697917,0.031250,0.062500,0.125000,0.250000
std,367.544609,1.092351,5.548926,4.584848,2.701071,3.525056,2.618520,1.731263,0.174448,0.242694,0.331584,0.434145
min,285.000000,23.920000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,670.500000,26.560000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,824.000000,27.225000,4.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1182.500000,28.000000,10.000000,6.250000,4.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.250000
max,1841.300000,29.780000,21.000000,19.000000,11.000000,16.000000,13.000000,8.000000,1.000000,1.000000,1.000000,1.000000


In [1254]:
kaggle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   version                          192 non-null    int64  
 1   team                             192 non-null    object 
 2   continent                        192 non-null    object 
 3   is_host                          192 non-null    int64  
 4   goals_scored_last_4y             192 non-null    int64  
 5   goals_received_last_4y           192 non-null    int64  
 6   wins_last_4y                     192 non-null    int64  
 7   losses_last_4y                   192 non-null    int64  
 8   draws_last_4y                    192 non-null    int64  
 9   world_cup_titles_before          192 non-null    int64  
 10  squad_total_market_value_eur     160 non-null    float64
 11  fifa_rank_pre_tournament         192 non-null    int64  
 12  fifa_points_pre_tourna

In [1255]:
kaggle_clean = kaggle[[
    'version',
    'team',
    'is_host',
    'goals_scored_last_4y',
    'goals_received_last_4y',
    'wins_last_4y',
    'losses_last_4y',
    'draws_last_4y',
    'world_cup_titles_before',
    'fifa_rank_pre_tournament',
    'fifa_points_pre_tournament',
    'squad_avg_age',
    'world_cup_participations_before',
    'groups_passed_before',
    'round16_before',
    'quarterfinals_before',
    'semifinals_before',
    'finals_before'
]].copy()

kaggle_clean['tournament_id'] = kaggle_clean['version'].map({2002:'WC-2002',2006:'WC-2006',2010:'WC-2010',
                                                             2014:'WC-2014',2018:'WC-2018',2022:'WC-2022'})

members = Members()
fifa_dict = {member.name: member.code for name, member in members.items()}
# manually fix country codes for teams not found in fifacodes
fifa_dict['China PR'] = 'CHN'
fifa_dict['Serbia and Montenegro'] = 'SCG'
kaggle_clean['team_code'] = kaggle_clean['team'].map(fifa_dict)

kaggle_clean = kaggle_clean.drop(columns = ['version'])

In [1256]:
# manually fill kaggle dataset for teams missing
# AGO (Angola) 2006, TGO (Togo) 2006, TTO (Trinidad) 2006, HND (Honduras) 2010/2014
missing_teams = pd.DataFrame([
    # Angola 2006 data
    {'tournament_id': 'WC-2006', 'team_code': 'AGO', 'is_host': 0, 
     'fifa_rank_pre_tournament': 57, 'fifa_points_pre_tournament': 581,
     # 2005, 2004, 2003, 2002
     'goals_scored_last_4y': 11+23+9+4,
     'goals_received_last_4y': 12+14+7+10,
     'wins_last_4y': 4+8+2+0,
     'losses_last_4y': 3+3+2+3,
     'draws_last_4y': 2+4+9+2,
     'world_cup_titles_before': 0},

     # Togo 2006 data
    {'tournament_id': 'WC-2006', 'team_code': 'TGO', 'is_host': 0, 
     'fifa_rank_pre_tournament': 61, 'fifa_points_pre_tournament': 569,
     # 2005, 2004, 2003, 2002
     'goals_scored_last_4y': 12+6+9+7,
     'goals_received_last_4y': 4+4+7+9,
     'wins_last_4y': 5+2+3+3,
     'losses_last_4y': 0+2+2+3,
     'draws_last_4y': 2+3+0+4,
     'world_cup_titles_before': 0},

     # Trinidad 2006 data
    {'tournament_id': 'WC-2006', 'team_code': 'TTO', 'is_host': 0, 
     'fifa_rank_pre_tournament': 47, 'fifa_points_pre_tournament': 604,
     # 2005, 2004, 2003, 2002
     'goals_scored_last_4y': 52+20+14+4,
     'goals_received_last_4y': 37+19+22+13,
     'wins_last_4y': 18+7+4+1,
     'losses_last_4y': 11+5+10+5,
     'draws_last_4y': 2+2+3+4,
     'world_cup_titles_before': 0},

     # Honduras 2010 data
    {'tournament_id': 'WC-2010', 'team_code': 'HND', 'is_host': 0, 
     'fifa_rank_pre_tournament': 38, 'fifa_points_pre_tournament': 734,
     # 2009, 2008, 2007, 2006
     'goals_scored_last_4y': 25+20+38+15,
     'goals_received_last_4y': 14+8+21+13,
     'wins_last_4y': 10+8+9+5,
     'losses_last_4y': 5+1+5+3,
     'draws_last_4y': 1+4+3+1,
     'world_cup_titles_before': 0},

     # Honduras 2014 data
    {'tournament_id': 'WC-2014', 'team_code': 'HND', 'is_host': 0, 
     'fifa_rank_pre_tournament': 33, 'fifa_points_pre_tournament': 731,
     # 2013, 2012, 2011, 2010
     'goals_scored_last_4y': 26+7+27+27,
     'goals_received_last_4y': 18+14+26+24,
     'wins_last_4y': 9+3+6+9,
     'losses_last_4y': 7+7+6+10,
     'draws_last_4y': 5+3+7+3,
     'world_cup_titles_before': 0},
])

kaggle_clean = pd.concat([kaggle_clean, missing_teams], ignore_index=True)

In [1257]:
kaggle_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   team                             192 non-null    object 
 1   is_host                          197 non-null    int64  
 2   goals_scored_last_4y             197 non-null    int64  
 3   goals_received_last_4y           197 non-null    int64  
 4   wins_last_4y                     197 non-null    int64  
 5   losses_last_4y                   197 non-null    int64  
 6   draws_last_4y                    197 non-null    int64  
 7   world_cup_titles_before          197 non-null    int64  
 8   fifa_rank_pre_tournament         197 non-null    int64  
 9   fifa_points_pre_tournament       197 non-null    float64
 10  squad_avg_age                    192 non-null    float64
 11  world_cup_participations_before  192 non-null    float64
 12  groups_passed_before  

In [1258]:
folder = 'data/data-csv/'

for file in os.listdir(folder):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(folder, file))
        print(f"\n{file}")
        print(df.shape)
        print(df.columns.tolist())


groups.csv
(159, 7)
['key_id', 'tournament_id', 'tournament_name', 'stage_number', 'stage_name', 'group_name', 'count_teams']

managers.csv
(475, 7)
['key_id', 'manager_id', 'family_name', 'given_name', 'female', 'country_name', 'manager_wikipedia_link']

qualified_teams.csv
(625, 8)
['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name', 'team_code', 'count_matches', 'performance']

teams.csv
(88, 14)
['key_id', 'team_id', 'team_name', 'team_code', 'mens_team', 'womens_team', 'federation_name', 'region_name', 'confederation_id', 'confederation_name', 'confederation_code', 'mens_team_wikipedia_link', 'womens_team_wikipedia_link', 'federation_wikipedia_link']

player_appearances.csv
(27432, 21)
['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name', 'match_date', 'stage_name', 'group_name', 'team_id', 'team_name', 'team_code', 'home_team', 'away_team', 'player_id', 'family_name', 'given_name', 'shirt_number', 'position_name', 'position_code', 'starter', 

In [1259]:
# load jfjelstul tables that are most useful
matches = pd.read_csv('data/data-csv/matches.csv')
squads = pd.read_csv('data/data-csv/squads.csv')
players = pd.read_csv('data/data-csv/players.csv')
player_appearances = pd.read_csv('data/data-csv/player_appearances.csv')
goals = pd.read_csv('data/data-csv/goals.csv')
bookings = pd.read_csv('data/data-csv/bookings.csv')
subs = pd.read_csv('data/data-csv/substitutions.csv')

In [1260]:
matches.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name',
       'stage_name', 'group_name', 'group_stage', 'knockout_stage', 'replayed',
       'replay', 'match_date', 'match_time', 'stadium_id', 'stadium_name',
       'city_name', 'country_name', 'home_team_id', 'home_team_name',
       'home_team_code', 'away_team_id', 'away_team_name', 'away_team_code',
       'score', 'home_team_score', 'away_team_score', 'home_team_score_margin',
       'away_team_score_margin', 'extra_time', 'penalty_shootout',
       'score_penalties', 'home_team_score_penalties',
       'away_team_score_penalties', 'result', 'home_team_win', 'away_team_win',
       'draw'],
      dtype='object')

In [1261]:
# trim tables to 2002 onwards and most useful features
recent_cups = [f"WC-{4 * x + 2002}" for x in range(6)]

matches_clean = (
    matches[matches['tournament_id'].isin(recent_cups)][[
        # identifiers
        'tournament_id',
        'match_id',
        'stage_name',
        'group_name',
        'group_stage',
        'knockout_stage',
        'match_date',

        # team info
        'home_team_id',
        'home_team_code',
        'away_team_id',
        'away_team_code',

        # scores and results
        'home_team_score',
        'away_team_score',
        'home_team_win',
        'away_team_win',
        'draw',
        'penalty_shootout'
]]
        .replace({'not applicable':pd.NA})
        .copy()
)

In [1262]:
matches_clean.tail()

,tournament_id,match_id,stage_name,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,penalty_shootout
1243,WC-2022,M-2022-60,quarter-finals,<NA>,0,1,2022-12-10,T-28,ENG,T-30,FRA,1,2,0,1,0,0
1244,WC-2022,M-2022-61,semi-finals,<NA>,0,1,2022-12-13,T-03,ARG,T-18,HRV,3,0,1,0,0,0
1245,WC-2022,M-2022-62,semi-finals,<NA>,0,1,2022-12-14,T-30,FRA,T-47,MAR,2,0,1,0,0,0
1246,WC-2022,M-2022-63,third-place match,<NA>,0,1,2022-12-17,T-18,HRV,T-47,MAR,2,1,1,0,0,0
1247,WC-2022,M-2022-64,final,<NA>,0,1,2022-12-18,T-03,ARG,T-30,FRA,3,3,1,0,0,1


In [1263]:
squads.shape

(13843, 12)

In [1264]:
squads.tail(10)

,key_id,tournament_id,tournament_name,team_id,team_name,team_code,player_id,family_name,given_name,shirt_number,position_name,position_code
13833,13834,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-34567,Lockyer,Tom,17,defender,DF
13834,13835,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-56664,Williams,Jonny,18,midfielder,MF
13835,13836,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-37173,Harris,Mark,19,forward,FW
13836,13837,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-74707,James,Daniel,20,forward,FW
13837,13838,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-92482,Davies,Adam,21,goal keeper,GK
13838,13839,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-20564,Thomas,Sorba,22,midfielder,MF
13839,13840,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-24982,Levitt,Dylan,23,midfielder,MF
13840,13841,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-02030,Cabango,Ben,24,defender,DF
13841,13842,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-07137,Colwill,Rubin,25,midfielder,MF
13842,13843,WC-2022,2022 FIFA Men's World Cup,T-85,Wales,WAL,P-84269,Smith,Matthew,26,midfielder,MF


In [1265]:
squads.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'team_id', 'team_name',
       'team_code', 'player_id', 'family_name', 'given_name', 'shirt_number',
       'position_name', 'position_code'],
      dtype='object')

In [1266]:
squads_clean = squads[squads['tournament_id'].isin(recent_cups)][[
    'tournament_id',
    'team_code',
    'player_id',
    'family_name',
    'given_name',
    'position_code'
]].copy()

In [1267]:
squads_clean.head()

,tournament_id,team_code,player_id,family_name,given_name,position_code
7238,WC-2002,ARG,P-56385,Burgos,Germán,GK
7239,WC-2002,ARG,P-83403,Ayala,Roberto,DF
7240,WC-2002,ARG,P-60914,Sorín,Juan Pablo,DF
7241,WC-2002,ARG,P-31684,Pochettino,Mauricio,DF
7242,WC-2002,ARG,P-88203,Almeyda,Matías,MF


In [1268]:
players.shape

(10401, 13)

In [1269]:
players.head()

,key_id,player_id,family_name,given_name,birth_date,female,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,player_wikipedia_link
0,1,P-35894,A'Court,Alan,1934-09-30,0,0,0,0,1,1,1958,https://en.wikipedia.org/wiki/Alan_A%27Court
1,2,P-29915,Aarønes,Ann Kristin,1973-01-19,1,0,0,1,1,2,"1995, 1999",https://en.wikipedia.org/wiki/Ann_Kristin_Aar%...
2,3,P-03484,Aaronson,Brenden,2000-10-22,0,0,0,0,1,1,2022,https://en.wikipedia.org/wiki/Brenden_Aaronson
3,4,P-04189,Abadzhiev,Stefan,1934-07-03,0,0,0,0,1,1,1966,https://en.wikipedia.org/wiki/Stefan_Abadzhiev
4,5,P-03523,Abalo,Jean-Paul,1975-06-26,0,0,1,0,0,1,2006,https://en.wikipedia.org/wiki/Jean-Paul_Abalo


In [1270]:
players.columns

Index(['key_id', 'player_id', 'family_name', 'given_name', 'birth_date',
       'female', 'goal_keeper', 'defender', 'midfielder', 'forward',
       'count_tournaments', 'list_tournaments', 'player_wikipedia_link'],
      dtype='object')

In [1271]:
players_clean = players[players['female'] == 0][[
    'player_id',
    'birth_date',
]].copy()

In [1272]:
players_clean.head()

,player_id,birth_date
0,P-35894,1934-09-30
2,P-03484,2000-10-22
3,P-04189,1934-07-03
4,P-03523,1975-06-26
6,P-35138,1978-08-03


In [1273]:
player_appearances.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'match_id', 'match_name',
       'match_date', 'stage_name', 'group_name', 'team_id', 'team_name',
       'team_code', 'home_team', 'away_team', 'player_id', 'family_name',
       'given_name', 'shirt_number', 'position_name', 'position_code',
       'starter', 'substitute'],
      dtype='object')

In [1274]:
player_appearances_clean = player_appearances[player_appearances['tournament_id'].isin(recent_cups)][[
    'tournament_id',
    'match_id',
    'team_code',
    'player_id',
    'starter',
    'substitute'
]].copy()

In [1275]:
player_appearances_clean.head(20)

,tournament_id,match_id,team_code,player_id,starter,substitute
11110,WC-2002,M-2002-01,FRA,P-56735,1,0
11111,WC-2002,M-2002-01,FRA,P-96540,1,0
11112,WC-2002,M-2002-01,FRA,P-80680,1,0
11113,WC-2002,M-2002-01,FRA,P-79380,1,0
11114,WC-2002,M-2002-01,FRA,P-00916,1,0
11115,WC-2002,M-2002-01,FRA,P-51395,1,0
11116,WC-2002,M-2002-01,FRA,P-56947,1,0
11117,WC-2002,M-2002-01,FRA,P-55991,1,0
11118,WC-2002,M-2002-01,FRA,P-40400,1,0
11119,WC-2002,M-2002-01,FRA,P-19907,1,0


In [1276]:
goals.columns

Index(['key_id', 'goal_id', 'tournament_id', 'tournament_name', 'match_id',
       'match_name', 'match_date', 'stage_name', 'group_name', 'team_id',
       'team_name', 'team_code', 'home_team', 'away_team', 'player_id',
       'family_name', 'given_name', 'shirt_number', 'player_team_id',
       'player_team_name', 'player_team_code', 'minute_label',
       'minute_regulation', 'minute_stoppage', 'match_period', 'own_goal',
       'penalty'],
      dtype='object')

In [1277]:
goals_clean = goals[goals['tournament_id'].isin(recent_cups)][[
    'goal_id',
    'tournament_id',
    'match_id',
    'player_id',
    'player_team_code',
    'minute_regulation',
    'minute_stoppage',
    'match_period',
    'own_goal',
    'penalty'
]].copy()

In [1278]:
goals_clean.head()

,goal_id,tournament_id,match_id,player_id,player_team_code,minute_regulation,minute_stoppage,match_period,own_goal,penalty
2076,G-2077,WC-2002,M-2002-01,P-82212,SEN,30,0,first half,0,0
2077,G-2078,WC-2002,M-2002-02,P-18804,CMR,39,0,first half,0,0
2078,G-2079,WC-2002,M-2002-02,P-00105,IRL,52,0,second half,0,0
2079,G-2080,WC-2002,M-2002-03,P-50534,DNK,45,0,first half,0,0
2080,G-2081,WC-2002,M-2002-03,P-49428,URY,47,0,second half,0,0


In [1279]:
bookings.columns

Index(['key_id', 'booking_id', 'tournament_id', 'tournament_name', 'match_id',
       'match_name', 'match_date', 'stage_name', 'group_name', 'team_id',
       'team_name', 'team_code', 'home_team', 'away_team', 'player_id',
       'family_name', 'given_name', 'shirt_number', 'minute_label',
       'minute_regulation', 'minute_stoppage', 'match_period', 'yellow_card',
       'red_card', 'second_yellow_card', 'sending_off'],
      dtype='object')

In [1280]:
bookings_clean = bookings[bookings['tournament_id'].isin(recent_cups)][[
    'booking_id',
    'tournament_id',
    'match_id',
    'team_code',
    'home_team',
    'away_team',
    'player_id',
    'minute_regulation',
    'minute_stoppage',
    'match_period',
    'yellow_card',
    'red_card',
    'second_yellow_card',
    'sending_off'
]].copy()

In [1281]:
bookings_clean.head()

,booking_id,tournament_id,match_id,team_code,home_team,away_team,player_id,minute_regulation,minute_stoppage,match_period,yellow_card,red_card,second_yellow_card,sending_off
1216,B-1217,WC-2002,M-2002-01,FRA,1,0,P-40400,45,2,"first half, stoppage time",1,0,0,0
1217,B-1218,WC-2002,M-2002-01,SEN,0,1,P-95302,51,0,second half,1,0,0,0
1218,B-1219,WC-2002,M-2002-02,IRL,1,0,P-15705,30,0,first half,1,0,0,0
1219,B-1220,WC-2002,M-2002-02,IRL,1,0,P-56778,51,0,second half,1,0,0,0
1220,B-1221,WC-2002,M-2002-02,IRL,1,0,P-19727,82,0,second half,1,0,0,0


In [1282]:
subs.columns

Index(['key_id', 'substitution_id', 'tournament_id', 'tournament_name',
       'match_id', 'match_name', 'match_date', 'stage_name', 'group_name',
       'team_id', 'team_name', 'team_code', 'home_team', 'away_team',
       'player_id', 'family_name', 'given_name', 'shirt_number',
       'minute_label', 'minute_regulation', 'minute_stoppage', 'match_period',
       'going_off', 'coming_on'],
      dtype='object')

In [1283]:
subs_clean = subs[subs['tournament_id'].isin(recent_cups)][[
    'substitution_id',
    'tournament_id',
    'match_id',
    'team_code',
    'home_team',
    'away_team',
    'player_id',
    'minute_regulation',
    'minute_stoppage',
    'match_period',
    'going_off',
    'coming_on'
]].copy()

In [1284]:
subs_clean.head()

,substitution_id,tournament_id,match_id,team_code,home_team,away_team,player_id,minute_regulation,minute_stoppage,match_period,going_off,coming_on
3230,S-3231,WC-2002,M-2002-01,FRA,1,0,P-80680,60,0,second half,1,0
3231,S-3232,WC-2002,M-2002-01,FRA,1,0,P-74065,60,0,second half,0,1
3232,S-3233,WC-2002,M-2002-01,FRA,1,0,P-00916,81,0,second half,1,0
3233,S-3234,WC-2002,M-2002-01,FRA,1,0,P-53533,81,0,second half,0,1
3234,S-3235,WC-2002,M-2002-02,IRL,1,0,P-15705,46,0,second half,1,0


In [1285]:
goals_clean.head()

,goal_id,tournament_id,match_id,player_id,player_team_code,minute_regulation,minute_stoppage,match_period,own_goal,penalty
2076,G-2077,WC-2002,M-2002-01,P-82212,SEN,30,0,first half,0,0
2077,G-2078,WC-2002,M-2002-02,P-18804,CMR,39,0,first half,0,0
2078,G-2079,WC-2002,M-2002-02,P-00105,IRL,52,0,second half,0,0
2079,G-2080,WC-2002,M-2002-03,P-50534,DNK,45,0,first half,0,0
2080,G-2081,WC-2002,M-2002-03,P-49428,URY,47,0,second half,0,0


In [ ]:
# fix discrepancies between Kaggle and FIFA codes for countries
code_map = {
    'DEU': 'GER',  # Germany
    'CHE': 'SUI',  # Switzerland
    'HRV': 'CRO',  # Croatia
    'NLD': 'NED',  # Netherlands
    'PRT': 'POR',  # Portugal
    'DZA': 'ALG',  # Algeria
    'CHL': 'CHI',  # Chile
    'DNK': 'DEN',  # Denmark
    'GRC': 'GRE',  # Greece
    'PRY': 'PAR',  # Paraguay
    'SAU': 'KSA',  # Saudi Arabia
    'URY': 'URU',  # Uruguay
    'ZAF': 'RSA',  # South Africa
    'CRI': 'CRC'   # Costa Rica
}

for df in [kaggle_clean, matches_clean, squads_clean, players_clean, player_appearances_clean, goals_clean, bookings_clean, subs_clean]:
    for col in ['home_team_code', 'away_team_code', 'team_code']:
        if col in df.columns:
            df[col] = df[col].replace(code_map)

In [ ]:
goals_agg = (goals_clean
    .groupby(['tournament_id', 'match_id', 'player_team_code'])
    .agg(
        goals_scored=('goal_id', 'count'),
        own_goals=('own_goal', 'sum'),
        penalties=('penalty', 'sum')
    )
    .reset_index()
)

goals_agg.head()

,tournament_id,match_id,player_team_code,goals_scored,own_goals,penalties
0,WC-2002,M-2002-01,SEN,1,0,0
1,WC-2002,M-2002-02,CMR,1,0,0
2,WC-2002,M-2002-02,IRL,1,0,0
3,WC-2002,M-2002-03,DNK,2,0,0
4,WC-2002,M-2002-03,URY,1,0,0


In [ ]:
match_table = (matches_clean
    .merge(
        goals_agg.rename(columns = {'player_team_code':'home_team_code', 'goals_scored': 'home_goals_scored',
                                  'own_goals':'home_own_goals', 'penalties': 'home_penalties'}),
        on = ['tournament_id','match_id','home_team_code'], how = 'left'
    )
    .merge(
        goals_agg.rename(columns = {'player_team_code':'away_team_code', 'goals_scored':'away_goals_scored',
                                  'own_goals':'away_own_goals', 'penalties':'away_penalties'}),
        on = ['tournament_id','match_id','away_team_code'], how = 'left'
    )
    .fillna(0)
)

match_table['group_name'] = match_table['group_name'].str[6:]

In [ ]:
match_table.head()

,tournament_id,match_id,stage_name,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,penalty_shootout,home_goals_scored,home_own_goals,home_penalties,away_goals_scored,away_own_goals,away_penalties
0,WC-2002,M-2002-01,group stage,A,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,1,0,1,0,0,0.0,0.0,0.0,1.0,0.0,0.0
1,WC-2002,M-2002-02,group stage,E,1,0,2002-06-01,T-60,IRL,T-11,CMR,1,1,0,0,1,0,1.0,0.0,0.0,1.0,0.0,0.0
2,WC-2002,M-2002-03,group stage,A,1,0,2002-06-01,T-84,URU,T-22,DEN,1,2,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,WC-2002,M-2002-04,group stage,E,1,0,2002-06-01,T-31,GER,T-63,KSA,8,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,WC-2002,M-2002-05,group stage,F,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,0,1,0,0,0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
kaggle_clean.head()

,team,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,fifa_rank_pre_tournament,fifa_points_pre_tournament,squad_avg_age,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,tournament_id,team_code
0,Angola,0,61,49,19,13,14,0,57,581.0,29.2,0.0,0.0,0.0,0.0,0.0,0.0,WC-2006,ANG
1,Argentina,0,97,55,31,10,10,2,9,746.0,27.8,13.0,10.0,5.0,6.0,4.0,4.0,WC-2006,ARG
2,Australia,0,101,34,23,8,5,0,42,612.0,27.1,1.0,0.0,0.0,0.0,0.0,0.0,WC-2006,AUS
3,Brazil,0,117,47,30,9,17,5,1,827.0,27.1,17.0,15.0,7.0,11.0,9.0,6.0,WC-2006,BRA
4,Costa Rica,0,89,84,26,25,11,0,26,683.0,24.7,2.0,1.0,1.0,0.0,0.0,0.0,WC-2006,CRC


In [ ]:
match_table = (match_table
    .merge(
        kaggle_clean[['tournament_id', 'team_code', 'is_host', 'goals_scored_last_4y', 'goals_received_last_4y',
                      'wins_last_4y', 'losses_last_4y', 'draws_last_4y', 'fifa_rank_pre_tournament',
                      'fifa_points_pre_tournament','world_cup_participations_before']]
        .rename(columns = {
            'team_code': 'home_team_code',
            'is_host': 'home_is_host',
            'goals_scored_last_4y': 'home_goals_scored_last_4y',
            'goals_received_last_4y': 'home_goals_received_last_4y',
            'wins_last_4y': 'home_wins_last_4y',
            'losses_last_4y': 'home_losses_last_4y',
            'draws_last_4y': 'home_draws_last_4y',
            'fifa_rank_pre_tournament': 'home_fifa_rank_pre_tournament',
            'fifa_points_pre_tournament': 'home_fifa_points_pre_tournament',
            'world_cup_participations_before':'home_world_cups_before'
        }),
        on = ['tournament_id', 'home_team_code'], how = 'left'
    )
    .merge(
        kaggle_clean[['tournament_id', 'team_code', 'is_host', 'goals_scored_last_4y', 'goals_received_last_4y',
                      'wins_last_4y', 'losses_last_4y', 'draws_last_4y', 'fifa_rank_pre_tournament',
                      'fifa_points_pre_tournament','world_cup_participations_before']]
        .rename(columns = {
            'team_code': 'away_team_code',
            'is_host': 'away_is_host',
            'goals_scored_last_4y': 'away_goals_scored_last_4y',
            'goals_received_last_4y': 'away_goals_received_last_4y',
            'wins_last_4y': 'away_wins_last_4y',
            'losses_last_4y': 'away_losses_last_4y',
            'draws_last_4y': 'away_draws_last_4y',
            'fifa_rank_pre_tournament': 'away_fifa_rank_pre_tournament',
            'fifa_points_pre_tournament': 'away_fifa_points_pre_tournament',
            'world_cup_participations_before':'away_world_cups_before'
        }),
        on = ['tournament_id', 'away_team_code'], how = 'left'
    )
)

In [ ]:
# drop columns that would create data leakage for training
match_table = match_table.drop(columns = ['stage_name','home_own_goals','home_penalties','away_own_goals',
                                          'away_penalties','penalty_shootout'])

# convert match date to year
match_table['match_date'] = pd.to_datetime(match_table['match_date'])
# calculate difference in fifa rank and points between teams
match_table['fifa_rank_diff'] = match_table['home_fifa_rank_pre_tournament'] - match_table['away_fifa_rank_pre_tournament']
match_table['fifa_pt_diff'] = match_table['home_fifa_points_pre_tournament'] - match_table['away_fifa_points_pre_tournament']

drop_before_training = ['tournament_id','match_id','home_team_id','home_team_code','away_team_id','away_team_code',
                        'match_date','home_team_win','draw','away_team_win','group_name','home_goals_scored',
                        'away_goals_scored','home_team_score','away_team_score']

In [ ]:
match_table.head()

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,home_goals_scored,away_goals_scored,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,home_world_cups_before,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,away_world_cups_before,fifa_rank_diff,fifa_pt_diff
0,WC-2002,M-2002-01,A,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,1,0,1,0,0.0,1.0,0,101,39,33,6,10,1,802.0,10.0,0,79,46,26,12,21,42,599.0,0.0,-41,203.0
1,WC-2002,M-2002-02,E,1,0,2002-06-01,T-60,IRL,T-11,CMR,1,1,0,0,1,1.0,1.0,0,66,28,21,7,9,15,674.0,2.0,0,65,25,23,5,13,17,672.0,4.0,-2,2.0
2,WC-2002,M-2002-03,A,1,0,2002-06-01,T-84,URU,T-22,DEN,1,2,0,1,0,0.0,0.0,0,56,50,18,15,12,24,652.0,9.0,0,62,38,20,10,11,20,657.0,2.0,4,-5.0
3,WC-2002,M-2002-04,E,1,0,2002-06-01,T-31,GER,T-63,KSA,8,0,1,0,0,0.0,0.0,0,100,58,24,13,10,11,695.0,14.0,0,156,73,48,17,13,34,627.0,2.0,-23,68.0
4,WC-2002,M-2002-05,F,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,0,1,0,0,1.0,0.0,0,70,34,23,5,10,3,784.0,12.0,0,71,36,26,7,12,27,644.0,2.0,-24,140.0


In [ ]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_name', 'group_stage',
       'knockout_stage', 'match_date', 'home_team_id', 'home_team_code',
       'away_team_id', 'away_team_code', 'home_team_score', 'away_team_score',
       'home_team_win', 'away_team_win', 'draw', 'home_goals_scored',
       'away_goals_scored', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'home_world_cups_before', 'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'away_world_cups_before', 'fifa_rank_diff', 'fifa_pt_diff'],
      dtype='object')

In [ ]:
player_appearances_clean.merge(players_clean, on = 'player_id').head()

,tournament_id,match_id,team_code,player_id,starter,substitute,birth_date
0,WC-2002,M-2002-01,FRA,P-56735,1,0,1969-12-09
1,WC-2002,M-2002-01,FRA,P-96540,1,0,1976-06-23
2,WC-2002,M-2002-01,FRA,P-80680,1,0,1968-03-09
3,WC-2002,M-2002-01,FRA,P-79380,1,0,1968-09-07
4,WC-2002,M-2002-01,FRA,P-00916,1,0,1974-05-10


In [ ]:
# calculate average age of each starting 11

# start with player_appearances_clean table, where only starters are included
squad_avg_age = (player_appearances_clean[player_appearances_clean['starter'] == 1]
    # add birthdates 
    .merge(players_clean, on = 'player_id', how = 'left')
    # add match_date
    .merge(matches_clean[['tournament_id', 'match_id', 'match_date']], on = ['tournament_id', 'match_id'], how = 'left')
    # calculate age by subtracting birthdate
    .assign(age = lambda df: (pd.to_datetime(df['match_date']) - pd.to_datetime(df['birth_date'])).dt.days // 365)
    # find the mean age for players in each team for each match
    .groupby(['tournament_id', 'match_id', 'team_code'])['age']
    .mean()
    .reset_index()
    .rename(columns = {'age': 'squad_avg_age'})
)

In [ ]:
squad_avg_age.head()

,tournament_id,match_id,team_code,squad_avg_age
0,WC-2002,M-2002-01,FRA,29.545455
1,WC-2002,M-2002-01,SEN,25.363636
2,WC-2002,M-2002-02,CMR,25.454545
3,WC-2002,M-2002-02,IRL,26.727273
4,WC-2002,M-2002-03,DEN,28.000000


In [ ]:
match_table = (match_table
    .merge(
        squad_avg_age.rename(columns = {
            'team_code': 'home_team_code',
            'squad_avg_age': 'home_squad_avg_age'
        }),
        on = ['tournament_id','match_id', 'home_team_code'], how = 'left'
    )
    .merge(
        squad_avg_age.rename(columns = {
            'team_code': 'away_team_code',
            'squad_avg_age': 'away_squad_avg_age'
        }),
        on = ['tournament_id','match_id', 'away_team_code'], how = 'left'
    )
)

# calculate squad age difference
match_table['squad_age_diff'] = match_table['home_squad_avg_age'] - match_table['away_squad_avg_age']

In [ ]:
match_table.head()

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,home_goals_scored,away_goals_scored,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,home_world_cups_before,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,away_world_cups_before,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff
0,WC-2002,M-2002-01,A,1,0,2002-05-31,T-30,FRA,T-65,SEN,0,1,0,1,0,0.0,1.0,0,101,39,33,6,10,1,802.0,10.0,0,79,46,26,12,21,42,599.0,0.0,-41,203.0,29.545455,25.363636,4.181818
1,WC-2002,M-2002-02,E,1,0,2002-06-01,T-60,IRL,T-11,CMR,1,1,0,0,1,1.0,1.0,0,66,28,21,7,9,15,674.0,2.0,0,65,25,23,5,13,17,672.0,4.0,-2,2.0,26.727273,25.454545,1.272727
2,WC-2002,M-2002-03,A,1,0,2002-06-01,T-84,URU,T-22,DEN,1,2,0,1,0,0.0,0.0,0,56,50,18,15,12,24,652.0,9.0,0,62,38,20,10,11,20,657.0,2.0,4,-5.0,26.181818,28.000000,-1.818182
3,WC-2002,M-2002-04,E,1,0,2002-06-01,T-31,GER,T-63,KSA,8,0,1,0,0,0.0,0.0,0,100,58,24,13,10,11,695.0,14.0,0,156,73,48,17,13,34,627.0,2.0,-23,68.0,27.181818,26.727273,0.454545
4,WC-2002,M-2002-05,F,1,0,2002-06-02,T-03,ARG,T-50,NGA,1,0,1,0,0,1.0,0.0,0,70,34,23,5,10,3,784.0,12.0,0,71,36,26,7,12,27,644.0,2.0,-24,140.0,28.000000,25.181818,2.818182


In [ ]:
home = match_table[match_table['group_stage'] == 1][
    ['tournament_id', 'match_id', 'match_date', 'home_team_code', 'home_team_win', 'draw']
].rename(columns = {
    'home_team_code':'team_code',
    'home_team_win':'win'
})

away = match_table[match_table['group_stage'] == 1][
    ['tournament_id', 'match_id', 'match_date', 'away_team_code', 'away_team_win', 'draw']
].rename(columns = {
    'away_team_code':'team_code',
    'away_team_win':'win'
})

team_group_matches = pd.concat([home, away]).sort_values(['tournament_id', 'team_code', 'match_date'])

team_group_matches['points'] = team_group_matches['win'] * 3 + team_group_matches['draw'] * 1

In [ ]:
team_group_matches['cumulative_points'] = (team_group_matches
    .groupby(['tournament_id', 'team_code'])['points']
    .transform(lambda x: x.shift(1).cumsum())
)

In [ ]:
match_table = (match_table
    .merge(
        team_group_matches[['tournament_id', 'match_id', 'team_code', 'cumulative_points']]
        .rename(columns = {
            'team_code': 'home_team_code',
            'cumulative_points': 'home_group_points'
        }),
        on = ['tournament_id', 'match_id', 'home_team_code'], how = 'left'
    )
    .merge(
        team_group_matches[['tournament_id', 'match_id', 'team_code', 'cumulative_points']]
        .rename(columns={
            'team_code': 'away_team_code',
            'cumulative_points': 'away_group_points'
        }),
        on = ['tournament_id', 'match_id', 'away_team_code'], how = 'left'
    )
)

In [ ]:
match_table.tail(10)

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,home_goals_scored,away_goals_scored,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,home_world_cups_before,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,away_world_cups_before,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points
374,WC-2022,M-2022-55,NaN,0,1,2022-12-06,T-47,MAR,T-73,ESP,0,0,1,0,0,0.0,0.0,0,89,30,32,6,12,22,1563.50,5.0,0,115,38,31,6,14,7,1715.22,15.0,15,-151.72,26.818182,25.727273,1.090909,NaN,NaN
375,WC-2022,M-2022-56,NaN,0,1,2022-12-06,T-58,POR,T-75,SUI,6,1,1,0,0,0.0,0.0,0,108,34,30,7,12,9,1676.56,7.0,0,90,57,22,15,13,15,1635.92,11.0,-6,40.64,26.818182,28.272727,-1.454545,NaN,NaN
376,WC-2022,M-2022-57,NaN,0,1,2022-12-09,T-18,CRO,T-09,BRA,1,1,1,0,0,0.0,1.0,0,82,64,24,14,11,12,1645.64,5.0,0,111,19,37,3,10,1,1841.30,21.0,11,-195.66,28.818182,28.090909,0.727273,NaN,NaN
377,WC-2022,M-2022-58,NaN,0,1,2022-12-09,T-48,NED,T-03,ARG,2,2,0,1,0,0.0,2.0,0,111,44,30,7,11,8,1694.51,10.0,0,98,27,33,4,13,3,1773.88,17.0,5,-79.37,27.000000,26.909091,0.090909,NaN,NaN
378,WC-2022,M-2022-59,NaN,0,1,2022-12-10,T-47,MAR,T-58,POR,1,0,1,0,0,1.0,0.0,0,89,30,32,6,12,22,1563.50,5.0,0,108,34,30,7,12,9,1676.56,7.0,13,-113.06,27.363636,26.363636,1.000000,NaN,NaN
379,WC-2022,M-2022-60,NaN,0,1,2022-12-10,T-28,ENG,T-30,FRA,1,2,0,1,0,1.0,2.0,0,122,34,33,8,10,5,1728.47,15.0,0,101,40,31,6,12,4,1759.78,15.0,1,-31.31,26.363636,27.363636,-1.000000,NaN,NaN
380,WC-2022,M-2022-61,NaN,0,1,2022-12-13,T-03,ARG,T-18,CRO,3,0,1,0,0,3.0,0.0,0,98,27,33,4,13,3,1773.88,17.0,0,82,64,24,14,11,12,1645.64,5.0,-9,128.24,27.181818,28.818182,-1.636364,NaN,NaN
381,WC-2022,M-2022-62,NaN,0,1,2022-12-14,T-30,FRA,T-47,MAR,2,0,1,0,0,2.0,0.0,0,101,40,31,6,12,4,1759.78,15.0,0,89,30,32,6,12,22,1563.50,5.0,-18,196.28,27.000000,26.909091,0.090909,NaN,NaN
382,WC-2022,M-2022-63,NaN,0,1,2022-12-17,T-18,CRO,T-47,MAR,2,1,1,0,0,0.0,1.0,0,82,64,24,14,11,12,1645.64,5.0,0,89,30,32,6,12,22,1563.50,5.0,-10,82.14,27.454545,26.181818,1.272727,NaN,NaN
383,WC-2022,M-2022-64,NaN,0,1,2022-12-18,T-03,ARG,T-30,FRA,3,3,1,0,0,3.0,3.0,0,98,27,33,4,13,3,1773.88,17.0,0,101,40,31,6,12,4,1759.78,15.0,-1,14.10,27.818182,27.545455,0.272727,NaN,NaN


In [ ]:
final_group_points = (team_group_matches
    .groupby(['tournament_id', 'team_code'])['points']
    .sum()
    .reset_index()
    .rename(columns={'points': 'final_group_points'})
)

In [ ]:
match_table = (match_table
    .merge(
        final_group_points.rename(columns = {
            'team_code': 'home_team_code',
            'final_group_points': 'home_final_group_points'
        }),
        on = ['tournament_id', 'home_team_code'], how = 'left'
    )
    .merge(
        final_group_points.rename(columns = {
            'team_code': 'away_team_code',
            'final_group_points': 'away_final_group_points'
        }),
        on = ['tournament_id', 'away_team_code'], how = 'left'
    )
)

In [ ]:
match_table['home_group_points'] = match_table['home_group_points'].fillna(
    match_table['home_final_group_points']
)
match_table['away_group_points'] = match_table['away_group_points'].fillna(
    match_table['away_final_group_points']
)
match_table = match_table.drop(columns = ['home_final_group_points', 'away_final_group_points'])

In [ ]:
match_table['group_pts_diff'] = match_table['home_group_points'] - match_table['away_group_points'] 

In [ ]:
match_table.tail()

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,home_goals_scored,away_goals_scored,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,home_world_cups_before,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,away_world_cups_before,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points,group_pts_diff
379,WC-2022,M-2022-60,NaN,0,1,2022-12-10,T-28,ENG,T-30,FRA,1,2,0,1,0,1.0,2.0,0,122,34,33,8,10,5,1728.47,15.0,0,101,40,31,6,12,4,1759.78,15.0,1,-31.31,26.363636,27.363636,-1.000000,7.0,6.0,1.0
380,WC-2022,M-2022-61,NaN,0,1,2022-12-13,T-03,ARG,T-18,CRO,3,0,1,0,0,3.0,0.0,0,98,27,33,4,13,3,1773.88,17.0,0,82,64,24,14,11,12,1645.64,5.0,-9,128.24,27.181818,28.818182,-1.636364,6.0,5.0,1.0
381,WC-2022,M-2022-62,NaN,0,1,2022-12-14,T-30,FRA,T-47,MAR,2,0,1,0,0,2.0,0.0,0,101,40,31,6,12,4,1759.78,15.0,0,89,30,32,6,12,22,1563.50,5.0,-18,196.28,27.000000,26.909091,0.090909,6.0,7.0,-1.0
382,WC-2022,M-2022-63,NaN,0,1,2022-12-17,T-18,CRO,T-47,MAR,2,1,1,0,0,0.0,1.0,0,82,64,24,14,11,12,1645.64,5.0,0,89,30,32,6,12,22,1563.50,5.0,-10,82.14,27.454545,26.181818,1.272727,5.0,7.0,-2.0
383,WC-2022,M-2022-64,NaN,0,1,2022-12-18,T-03,ARG,T-30,FRA,3,3,1,0,0,3.0,3.0,0,98,27,33,4,13,3,1773.88,17.0,0,101,40,31,6,12,4,1759.78,15.0,-1,14.10,27.818182,27.545455,0.272727,6.0,6.0,0.0


In [ ]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_name', 'group_stage',
       'knockout_stage', 'match_date', 'home_team_id', 'home_team_code',
       'away_team_id', 'away_team_code', 'home_team_score', 'away_team_score',
       'home_team_win', 'away_team_win', 'draw', 'home_goals_scored',
       'away_goals_scored', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'home_world_cups_before', 'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'away_world_cups_before', 'fifa_rank_diff', 'fifa_pt_diff',
       'home_squad_avg_age', 'away_squad_avg_age', 'squad_age_diff',
       'home_group_points', 'away_group_points', 'grou

In [ ]:

home = match_table[['tournament_id', 'match_id', 'match_date', 'home_team_code',
                    'home_team_score', 'away_team_score']].rename(columns = {
    'home_team_code': 'team_code',
    'home_team_score': 'goals_scored',
    'away_team_score': 'goals_conceded'
})

away = match_table[['tournament_id', 'match_id', 'match_date', 'away_team_code',
                    'away_team_score', 'home_team_score']].rename(columns = {
    'away_team_code': 'team_code',
    'away_team_score': 'goals_scored',
    'home_team_score': 'goals_conceded'
})

team_match_goals = pd.concat([home, away]).sort_values(['tournament_id', 'team_code', 'match_date'])

In [ ]:
team_match_goals.head()

,tournament_id,match_id,match_date,team_code,goals_scored,goals_conceded
4,WC-2002,M-2002-05,2002-06-02,ARG,1,0
22,WC-2002,M-2002-23,2002-06-07,ARG,0,1
37,WC-2002,M-2002-38,2002-06-12,ARG,1,1
12,WC-2002,M-2002-13,2002-06-04,BEL,2,2
30,WC-2002,M-2002-31,2002-06-10,BEL,1,1


In [ ]:
team_match_goals['rolling_avg_goals_scored'] = (team_match_goals
    .groupby(['tournament_id', 'team_code'])['goals_scored']
    .transform(lambda x: x.shift(1).expanding().mean())
)

team_match_goals['rolling_avg_goals_conceded'] = (team_match_goals
    .groupby(['tournament_id', 'team_code'])['goals_conceded']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [ ]:
team_match_goals.head(20)

,tournament_id,match_id,match_date,team_code,goals_scored,goals_conceded,rolling_avg_goals_scored,rolling_avg_goals_conceded
4,WC-2002,M-2002-05,2002-06-02,ARG,1,0,NaN,NaN
22,WC-2002,M-2002-23,2002-06-07,ARG,0,1,1.000000,0.000000
37,WC-2002,M-2002-38,2002-06-12,ARG,1,1,0.500000,0.500000
12,WC-2002,M-2002-13,2002-06-04,BEL,2,2,NaN,NaN
30,WC-2002,M-2002-31,2002-06-10,BEL,1,1,2.000000,2.000000
44,WC-2002,M-2002-45,2002-06-14,BEL,3,2,1.500000,1.500000
53,WC-2002,M-2002-54,2002-06-17,BEL,0,2,2.000000,1.666667
9,WC-2002,M-2002-10,2002-06-03,BRA,2,1,NaN,NaN
25,WC-2002,M-2002-26,2002-06-08,BRA,4,0,2.000000,1.000000
40,WC-2002,M-2002-41,2002-06-13,BRA,5,2,3.000000,0.500000


In [ ]:
match_table = (match_table
    .merge(
        team_match_goals[['tournament_id', 'match_id', 'team_code', 
                          'rolling_avg_goals_scored', 'rolling_avg_goals_conceded']]
        .rename(columns = {
            'team_code': 'home_team_code',
            'rolling_avg_goals_scored': 'home_rolling_avg_goals_scored',
            'rolling_avg_goals_conceded': 'home_rolling_avg_goals_conceded'
        }),
        on = ['tournament_id', 'match_id', 'home_team_code'], how = 'left'
    )
    .merge(
        team_match_goals[['tournament_id', 'match_id', 'team_code',
                          'rolling_avg_goals_scored', 'rolling_avg_goals_conceded']]
        .rename(columns = {
            'team_code': 'away_team_code',
            'rolling_avg_goals_scored': 'away_rolling_avg_goals_scored',
            'rolling_avg_goals_conceded': 'away_rolling_avg_goals_conceded'
        }),
        on = ['tournament_id', 'match_id', 'away_team_code'], how = 'left'
    )
)

In [ ]:
match_table.tail()

,tournament_id,match_id,group_name,group_stage,knockout_stage,match_date,home_team_id,home_team_code,away_team_id,away_team_code,home_team_score,away_team_score,home_team_win,away_team_win,draw,home_goals_scored,away_goals_scored,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,home_world_cups_before,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,away_world_cups_before,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points,group_pts_diff,home_rolling_avg_goals_scored,home_rolling_avg_goals_conceded,away_rolling_avg_goals_scored,away_rolling_avg_goals_conceded
379,WC-2022,M-2022-60,NaN,0,1,2022-12-10,T-28,ENG,T-30,FRA,1,2,0,1,0,1.0,2.0,0,122,34,33,8,10,5,1728.47,15.0,0,101,40,31,6,12,4,1759.78,15.0,1,-31.31,26.363636,27.363636,-1.000000,7.0,6.0,1.0,3.0,0.500000,2.250000,1.000000
380,WC-2022,M-2022-61,NaN,0,1,2022-12-13,T-03,ARG,T-18,CRO,3,0,1,0,0,3.0,0.0,0,98,27,33,4,13,3,1773.88,17.0,0,82,64,24,14,11,12,1645.64,5.0,-9,128.24,27.181818,28.818182,-1.636364,6.0,5.0,1.0,1.8,1.000000,1.200000,0.600000
381,WC-2022,M-2022-62,NaN,0,1,2022-12-14,T-30,FRA,T-47,MAR,2,0,1,0,0,2.0,0.0,0,101,40,31,6,12,4,1759.78,15.0,0,89,30,32,6,12,22,1563.50,5.0,-18,196.28,27.000000,26.909091,0.090909,6.0,7.0,-1.0,2.2,1.000000,1.000000,0.200000
382,WC-2022,M-2022-63,NaN,0,1,2022-12-17,T-18,CRO,T-47,MAR,2,1,1,0,0,0.0,1.0,0,82,64,24,14,11,12,1645.64,5.0,0,89,30,32,6,12,22,1563.50,5.0,-10,82.14,27.454545,26.181818,1.272727,5.0,7.0,-2.0,1.0,1.000000,0.833333,0.500000
383,WC-2022,M-2022-64,NaN,0,1,2022-12-18,T-03,ARG,T-30,FRA,3,3,1,0,0,3.0,3.0,0,98,27,33,4,13,3,1773.88,17.0,0,101,40,31,6,12,4,1759.78,15.0,-1,14.10,27.818182,27.545455,0.272727,6.0,6.0,0.0,2.0,0.833333,2.166667,0.833333


In [ ]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_name', 'group_stage',
       'knockout_stage', 'match_date', 'home_team_id', 'home_team_code',
       'away_team_id', 'away_team_code', 'home_team_score', 'away_team_score',
       'home_team_win', 'away_team_win', 'draw', 'home_goals_scored',
       'away_goals_scored', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'home_world_cups_before', 'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'away_world_cups_before', 'fifa_rank_diff', 'fifa_pt_diff',
       'home_squad_avg_age', 'away_squad_avg_age', 'squad_age_diff',
       'home_group_points', 'away_group_points', 'grou

In [ ]:
match_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   tournament_id                    384 non-null    object        
 1   match_id                         384 non-null    object        
 2   group_name                       288 non-null    object        
 3   group_stage                      384 non-null    int64         
 4   knockout_stage                   384 non-null    int64         
 5   match_date                       384 non-null    datetime64[ns]
 6   home_team_id                     384 non-null    object        
 7   home_team_code                   384 non-null    object        
 8   away_team_id                     384 non-null    object        
 9   away_team_code                   384 non-null    object        
 10  home_team_score                  384 non-null    int64        

In [ ]:
drop_before_training

['tournament_id',
 'match_id',
 'home_team_id',
 'home_team_code',
 'away_team_id',
 'away_team_code',
 'match_date',
 'home_team_win',
 'draw',
 'away_team_win',
 'group_name',
 'home_goals_scored',
 'away_goals_scored',
 'home_team_score',
 'away_team_score']

In [ ]:
X = match_table.drop(columns = drop_before_training)
Y = match_table[['home_team_win','away_team_win','draw']].idxmax(axis = 1)

In [ ]:
# fill NA for rolling averages with the average from the last 4 years pre-tournament
X['home_rolling_avg_goals_scored'] = X['home_rolling_avg_goals_scored'].fillna(
    X['home_goals_scored_last_4y'] / (X['home_wins_last_4y'] + X['home_losses_last_4y'] + X['home_draws_last_4y'])
)
X['home_rolling_avg_goals_conceded'] = X['home_rolling_avg_goals_conceded'].fillna(
    X['home_goals_received_last_4y'] / (X['home_wins_last_4y'] + X['home_losses_last_4y'] + X['home_draws_last_4y'])
)

# same for away
X['away_rolling_avg_goals_scored'] = X['away_rolling_avg_goals_scored'].fillna(
    X['away_goals_scored_last_4y'] / (X['away_wins_last_4y'] + X['away_losses_last_4y'] + X['away_draws_last_4y'])
)
X['away_rolling_avg_goals_conceded'] = X['away_rolling_avg_goals_conceded'].fillna(
    X['away_goals_received_last_4y'] / (X['away_wins_last_4y'] + X['away_losses_last_4y'] + X['away_draws_last_4y'])
)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2, random_state = 723)

model = RandomForestClassifier(n_estimators = 20, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

away_team_win       0.61      0.77      0.68        26
         draw       0.40      0.12      0.19        16
home_team_win       0.69      0.77      0.73        35

     accuracy                           0.64        77
    macro avg       0.57      0.56      0.53        77
 weighted avg       0.60      0.64      0.60        77



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2, random_state = 723)

model = RandomForestClassifier(n_estimators = 100, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

away_team_win       0.57      0.65      0.61        26
         draw       0.00      0.00      0.00        16
home_team_win       0.60      0.80      0.68        35

     accuracy                           0.58        77
    macro avg       0.39      0.48      0.43        77
 weighted avg       0.46      0.58      0.52        77



/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera

In [ ]:
X_train = X[match_table['tournament_id'] != 'WC-2022']
X_test = X[match_table['tournament_id'] == 'WC-2022']
y_train = Y[match_table['tournament_id'] != 'WC-2022']
y_test = Y[match_table['tournament_id'] == 'WC-2022']

model = RandomForestClassifier(n_estimators = 30, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

away_team_win       0.48      0.68      0.57        22
         draw       0.33      0.10      0.15        10
home_team_win       0.73      0.69      0.71        32

     accuracy                           0.59        64
    macro avg       0.52      0.49      0.48        64
 weighted avg       0.59      0.59      0.57        64



In [ ]:
X['rank_closeness'] = 1 / (abs(match_table['fifa_rank_diff']) + 1)
X['pts_closeness'] = 1 / (abs(match_table['fifa_pt_diff']) + 1)

X_train = X[match_table['tournament_id'] != 'WC-2022']
X_test = X[match_table['tournament_id'] == 'WC-2022']
y_train = Y[match_table['tournament_id'] != 'WC-2022']
y_test = Y[match_table['tournament_id'] == 'WC-2022']

model = RandomForestClassifier(n_estimators = 100, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('n_estimators = 100:\n',classification_report(y_test, y_pred))

model = RandomForestClassifier(n_estimators = 50, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('n_estimators = 50:\n',classification_report(y_test, y_pred))

model = RandomForestClassifier(n_estimators = 30, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('n_estimators = 30:\n',classification_report(y_test, y_pred))

model = RandomForestClassifier(n_estimators = 20, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('n_estimators = 20:\n',classification_report(y_test, y_pred))

/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera

n_estimators = 100:
                precision    recall  f1-score   support

away_team_win       0.50      0.68      0.58        22
         draw       0.00      0.00      0.00        10
home_team_win       0.65      0.69      0.67        32

     accuracy                           0.58        64
    macro avg       0.38      0.46      0.41        64
 weighted avg       0.50      0.58      0.53        64



/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/tonyianniello/opt/anaconda3/envs/basic/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(avera

n_estimators = 50:
                precision    recall  f1-score   support

away_team_win       0.48      0.68      0.57        22
         draw       0.00      0.00      0.00        10
home_team_win       0.67      0.69      0.68        32

     accuracy                           0.58        64
    macro avg       0.38      0.46      0.41        64
 weighted avg       0.50      0.58      0.53        64

n_estimators = 30:
                precision    recall  f1-score   support

away_team_win       0.48      0.68      0.57        22
         draw       0.33      0.10      0.15        10
home_team_win       0.73      0.69      0.71        32

     accuracy                           0.59        64
    macro avg       0.52      0.49      0.48        64
 weighted avg       0.59      0.59      0.57        64

n_estimators = 20:
                precision    recall  f1-score   support

away_team_win       0.52      0.59      0.55        22
         draw       0.20      0.10      0.13        1

In [ ]:
kaggle_2026 = pd.read_csv('data/test.csv')[[
    'version',
    'team',
    'is_host',
    'goals_scored_last_4y',
    'goals_received_last_4y',
    'wins_last_4y',
    'losses_last_4y',
    'draws_last_4y',
    'world_cup_titles_before',
    'fifa_rank_pre_tournament',
    'fifa_points_pre_tournament',
    'squad_avg_age',
    'world_cup_participations_before'
]]

In [ ]:
kaggle_2026['version'] = kaggle_2026['version'].replace(2026,'WC-2026')
kaggle_2026['team_code'] = (
    kaggle_2026['team']
    .map(fifa_dict)
    .fillna('CUW')
)

In [ ]:
kaggle_2026.head()

,version,team,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,fifa_rank_pre_tournament,fifa_points_pre_tournament,squad_avg_age,world_cup_participations_before,team_code
0,WC-2026,France,0,85,32,25,6,7,2,1,1877.32,26.7,16,FRA
1,WC-2026,Spain,0,104,32,29,2,8,1,2,1876.40,27.2,16,ESP
2,WC-2026,Argentina,0,80,14,30,4,3,3,3,1874.81,27.9,18,ARG
3,WC-2026,England,0,82,23,26,6,7,1,4,1825.97,26.8,16,ENG
4,WC-2026,Portugal,0,98,31,26,5,7,0,5,1763.83,27.1,8,POR


In [ ]:
X.columns

Index(['group_stage', 'knockout_stage', 'home_is_host',
       'home_goals_scored_last_4y', 'home_goals_received_last_4y',
       'home_wins_last_4y', 'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'home_world_cups_before', 'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'away_world_cups_before', 'fifa_rank_diff', 'fifa_pt_diff',
       'home_squad_avg_age', 'away_squad_avg_age', 'squad_age_diff',
       'home_group_points', 'away_group_points', 'group_pts_diff',
       'home_rolling_avg_goals_scored', 'home_rolling_avg_goals_conceded',
       'away_rolling_avg_goals_scored', 'away_rolling_avg_goals_conceded',
       'rank_closeness', 'pts_closeness'],
      dtype='object')

In [ ]:
matches_2026 = pd.read_csv('data/areezvisram12/matches.csv')[['id','match_number','home_team_id','away_team_id','stage_id','match_label']]
matches_2026['group_stage'] = (matches_2026['stage_id'] == 1).astype(int)
matches_2026['knockout_stage'] = (matches_2026['stage_id'] != 1).astype(int)
matches_2026['version'] = 'WC-2026'

In [ ]:
matches_2026.head()

,id,match_number,home_team_id,away_team_id,stage_id,match_label,group_stage,knockout_stage,version
0,1,1,1.0,2.0,1,Group A,1,0,WC-2026
1,2,2,3.0,4.0,1,Group A,1,0,WC-2026
2,3,3,5.0,6.0,1,Group B,1,0,WC-2026
3,4,4,13.0,14.0,1,Group D,1,0,WC-2026
4,5,5,7.0,8.0,1,Group B,1,0,WC-2026


In [ ]:
teams_2026 = pd.read_csv('data/areezvisram12/teams.csv')

In [ ]:
teams_2026[teams_2026['is_placeholder'] == True]

,id,team_name,fifa_code,group_letter,is_placeholder
3,4,Winner UEFA Playoff D,UEPD,A,True
5,6,Winner UEFA Playoff A,UEPA,B,True
15,16,Winner UEFA Playoff C,UEPC,D,True
22,23,Winner UEFA Playoff B,UEPB,F,True
34,35,Winner FIFA Playoff 2,FP02,I,True
41,42,Winner FIFA Playoff 1,FP01,K,True


In [ ]:
teams_2026 = teams_2026.replace({
    'UEPD':'CZE',
    'UEPA':'BIH',
    'UEPC':'TUR',
    'UEPB':'SWE',
    'FP02':'IRQ',
    'FP01':'COD'
})

In [ ]:
teams_2026 = teams_2026.drop(columns = 'is_placeholder')

In [ ]:
teams_2026.head()

,id,team_name,fifa_code,group_letter
0,1,Mexico,MEX,A
1,2,South Africa,RSA,A
2,3,South Korea,KOR,A
3,4,Winner UEFA Playoff D,CZE,A
4,5,Canada,CAN,B


In [ ]:
# add FIFA codes for home and away teams
matches_2026 = (matches_2026
                .merge(
                    teams_2026[['fifa_code','id']].rename(columns = {
                    'fifa_code':'home_team_code',
                    'id':'home_team_id'
                }),
                on = 'home_team_id', how = 'left'
                )
                .merge(
                    teams_2026[['fifa_code','id']].rename(columns = {
                    'fifa_code':'away_team_code',
                    'id':'away_team_id'
                }),
                on = 'away_team_id', how = 'left' 
                )
)

In [ ]:
matches_2026.head(10)

,id,match_number,home_team_id,away_team_id,stage_id,match_label,group_stage,knockout_stage,version,home_team_code,away_team_code
0,1,1,1.0,2.0,1,Group A,1,0,WC-2026,MEX,RSA
1,2,2,3.0,4.0,1,Group A,1,0,WC-2026,KOR,CZE
2,3,3,5.0,6.0,1,Group B,1,0,WC-2026,CAN,BIH
3,4,4,13.0,14.0,1,Group D,1,0,WC-2026,USA,PAR
4,5,5,7.0,8.0,1,Group B,1,0,WC-2026,QAT,SUI
5,6,6,9.0,10.0,1,Group C,1,0,WC-2026,BRA,MAR
6,7,7,11.0,12.0,1,Group C,1,0,WC-2026,HAI,SCO
7,8,8,15.0,16.0,1,Group D,1,0,WC-2026,AUS,TUR
8,9,9,17.0,18.0,1,Group E,1,0,WC-2026,GER,CUR
9,10,10,21.0,22.0,1,Group F,1,0,WC-2026,NED,JPN


In [ ]:
matches_2026.columns

Index(['id', 'match_number', 'home_team_id', 'away_team_id', 'stage_id',
       'match_label', 'group_stage', 'knockout_stage', 'version',
       'home_team_code', 'away_team_code'],
      dtype='object')

In [ ]:
match_table.columns

Index(['tournament_id', 'match_id', 'group_name', 'group_stage',
       'knockout_stage', 'match_date', 'home_team_id', 'home_team_code',
       'away_team_id', 'away_team_code', 'home_team_score', 'away_team_score',
       'home_team_win', 'away_team_win', 'draw', 'home_goals_scored',
       'away_goals_scored', 'home_is_host', 'home_goals_scored_last_4y',
       'home_goals_received_last_4y', 'home_wins_last_4y',
       'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'home_world_cups_before', 'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'away_world_cups_before', 'fifa_rank_diff', 'fifa_pt_diff',
       'home_squad_avg_age', 'away_squad_avg_age', 'squad_age_diff',
       'home_group_points', 'away_group_points', 'grou

In [ ]:
# join data from kaggle 2026 team dataset
matches_2026 = (matches_2026
                # home team data
                .merge(
                    kaggle_2026[['team_code','is_host','goals_scored_last_4y','goals_received_last_4y',
                                 'wins_last_4y','losses_last_4y','draws_last_4y','fifa_rank_pre_tournament',
                                 'fifa_points_pre_tournament','squad_avg_age','world_cup_participations_before']].rename(columns = {
                        'team_code':'home_team_code',
                        'is_host':'home_is_host',
                        'goals_scored_last_4y':'home_goals_scored_last_4y',
                        'goals_received_last_4y':'home_goals_received_last_4y',
                        'wins_last_4y':'home_wins_last_4y',
                        'losses_last_4y':'home_losses_last_4y',
                        'draws_last_4y':'home_draws_last_4y',
                        'fifa_rank_pre_tournament':'home_fifa_rank_pre_tournament',
                        'fifa_points_pre_tournament':'home_fifa_points_pre_tournament',
                        'squad_avg_age':'home_squad_avg_age',
                        'world_cup_participations_before':'home_world_cups_before'
                    }),
                    on = 'home_team_code', how = 'left'
                )
                # away team data
                .merge(
                    kaggle_2026[['team_code','is_host','goals_scored_last_4y','goals_received_last_4y',
                                 'wins_last_4y','losses_last_4y','draws_last_4y','fifa_rank_pre_tournament',
                                 'fifa_points_pre_tournament','squad_avg_age','world_cup_participations_before']].rename(columns = {
                        'team_code':'away_team_code',
                        'is_host':'away_is_host',
                        'goals_scored_last_4y':'away_goals_scored_last_4y',
                        'goals_received_last_4y':'away_goals_received_last_4y',
                        'wins_last_4y':'away_wins_last_4y',
                        'losses_last_4y':'away_losses_last_4y',
                        'draws_last_4y':'away_draws_last_4y',
                        'fifa_rank_pre_tournament':'away_fifa_rank_pre_tournament',
                        'fifa_points_pre_tournament':'away_fifa_points_pre_tournament',
                        'squad_avg_age':'away_squad_avg_age',
                        'world_cup_participations_before':'away_world_cups_before'
                    }),
                    on = 'away_team_code', how = 'left'
                )
)

matches_2026['fifa_rank_diff'] = matches_2026['home_fifa_rank_pre_tournament'] - matches_2026['away_fifa_rank_pre_tournament']
matches_2026['fifa_pt_diff'] = matches_2026['home_fifa_points_pre_tournament'] - matches_2026['away_fifa_points_pre_tournament']
matches_2026['squad_age_diff'] = matches_2026['home_squad_avg_age'] - matches_2026['away_squad_avg_age']
matches_2026['pts_closeness'] = 1 / (abs(matches_2026['fifa_rank_diff']) + 1)
matches_2026['rank_closeness'] = 1 / (abs(matches_2026['fifa_pt_diff']) + 1)
total_matches = matches_2026['home_wins_last_4y'] + matches_2026['home_losses_last_4y'] + matches_2026['home_draws_last_4y']
away_total = matches_2026['away_wins_last_4y'] + matches_2026['away_losses_last_4y'] + matches_2026['away_draws_last_4y']

# group points start at 0 for every team
matches_2026['home_group_points'] = 0
matches_2026['away_group_points'] = 0
matches_2026['group_pts_diff'] = 0

# use the average from the last 4 years to impute rolling goals scored and conceded
matches_2026['home_rolling_avg_goals_scored'] = matches_2026['home_goals_scored_last_4y'] / total_matches
matches_2026['home_rolling_avg_goals_conceded'] = matches_2026['home_goals_received_last_4y'] / total_matches
matches_2026['away_rolling_avg_goals_scored'] = matches_2026['away_goals_scored_last_4y'] / away_total
matches_2026['away_rolling_avg_goals_conceded'] = matches_2026['away_goals_received_last_4y'] / away_total

In [ ]:
pd.set_option('display.max_columns', None)
matches_2026.head()

,id,match_number,home_team_id,away_team_id,stage_id,match_label,group_stage,knockout_stage,version,home_team_code,away_team_code,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,home_squad_avg_age,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,away_squad_avg_age,fifa_rank_diff,fifa_pt_diff,squad_age_diff,pts_closeness,rank_closeness,home_group_points,away_group_points,group_pts_diff,home_rolling_avg_goals_scored,home_rolling_avg_goals_conceded,away_rolling_avg_goals_scored,away_rolling_avg_goals_conceded
0,1,1,1.0,2.0,1,Group A,1,0,WC-2026,MEX,RSA,1.0,80.0,52.0,27.0,12.0,15.0,15.0,1681.03,25.6,0.0,69.0,41.0,20.0,7.0,20.0,60.0,1429.73,26.3,-45.0,251.30,-0.7,0.021739,0.003964,0,0,0,1.481481,0.962963,1.468085,0.872340
1,2,2,3.0,4.0,1,Group A,1,0,WC-2026,KOR,CZE,0.0,76.0,36.0,21.0,6.0,12.0,25.0,1588.66,27.5,0.0,66.0,38.0,18.0,6.0,11.0,41.0,1501.38,27.1,-16.0,87.28,0.4,0.058824,0.011328,0,0,0,1.948718,0.923077,1.885714,1.085714
2,3,3,5.0,6.0,1,Group B,1,0,WC-2026,CAN,BIH,1.0,59.0,40.0,18.0,8.0,15.0,30.0,1556.48,24.6,0.0,38.0,55.0,9.0,16.0,6.0,65.0,1385.84,26.9,-35.0,170.64,-2.3,0.027778,0.005826,0,0,0,1.439024,0.975610,1.225806,1.774194
3,4,4,13.0,14.0,1,Group D,1,0,WC-2026,USA,PAR,1.0,101.0,60.0,26.0,17.0,9.0,16.0,1673.13,26.0,0.0,29.0,33.0,11.0,12.0,9.0,40.0,1503.50,28.3,-24.0,169.63,-2.3,0.040000,0.005861,0,0,0,1.942308,1.153846,0.906250,1.031250
4,5,5,7.0,8.0,1,Group B,1,0,WC-2026,QAT,SUI,0.0,74.0,70.0,19.0,16.0,13.0,55.0,1454.96,27.8,0.0,71.0,40.0,15.0,6.0,16.0,19.0,1649.40,27.5,36.0,-194.44,0.3,0.027027,0.005117,0,0,0,1.541667,1.458333,1.918919,1.081081


In [ ]:
# new df with the same column orientation as training data
X_2026 = matches_2026[X_train.columns]

KeyError: "['home_world_cups_before', 'away_world_cups_before'] not in index"

In [ ]:
X_2026.head()

,group_stage,knockout_stage,home_is_host,home_goals_scored_last_4y,home_goals_received_last_4y,home_wins_last_4y,home_losses_last_4y,home_draws_last_4y,home_fifa_rank_pre_tournament,home_fifa_points_pre_tournament,away_is_host,away_goals_scored_last_4y,away_goals_received_last_4y,away_wins_last_4y,away_losses_last_4y,away_draws_last_4y,away_fifa_rank_pre_tournament,away_fifa_points_pre_tournament,fifa_rank_diff,fifa_pt_diff,home_squad_avg_age,away_squad_avg_age,squad_age_diff,home_group_points,away_group_points,group_pts_diff,home_rolling_avg_goals_scored,home_rolling_avg_goals_conceded,away_rolling_avg_goals_scored,away_rolling_avg_goals_conceded,rank_closeness,pts_closeness
0,1,0,1.0,80.0,52.0,27.0,12.0,15.0,15.0,1681.03,0.0,69.0,41.0,20.0,7.0,20.0,60.0,1429.73,-45.0,251.30,25.6,26.3,-0.7,0,0,0,1.481481,0.962963,1.468085,0.872340,0.003964,0.021739
1,1,0,0.0,76.0,36.0,21.0,6.0,12.0,25.0,1588.66,0.0,66.0,38.0,18.0,6.0,11.0,41.0,1501.38,-16.0,87.28,27.5,27.1,0.4,0,0,0,1.948718,0.923077,1.885714,1.085714,0.011328,0.058824
2,1,0,1.0,59.0,40.0,18.0,8.0,15.0,30.0,1556.48,0.0,38.0,55.0,9.0,16.0,6.0,65.0,1385.84,-35.0,170.64,24.6,26.9,-2.3,0,0,0,1.439024,0.975610,1.225806,1.774194,0.005826,0.027778
3,1,0,1.0,101.0,60.0,26.0,17.0,9.0,16.0,1673.13,0.0,29.0,33.0,11.0,12.0,9.0,40.0,1503.50,-24.0,169.63,26.0,28.3,-2.3,0,0,0,1.942308,1.153846,0.906250,1.031250,0.005861,0.040000
4,1,0,0.0,74.0,70.0,19.0,16.0,13.0,55.0,1454.96,0.0,71.0,40.0,15.0,6.0,16.0,19.0,1649.40,36.0,-194.44,27.8,27.5,0.3,0,0,0,1.541667,1.458333,1.918919,1.081081,0.005117,0.027027


In [ ]:
X_train.columns

Index(['group_stage', 'knockout_stage', 'home_is_host',
       'home_goals_scored_last_4y', 'home_goals_received_last_4y',
       'home_wins_last_4y', 'home_losses_last_4y', 'home_draws_last_4y',
       'home_fifa_rank_pre_tournament', 'home_fifa_points_pre_tournament',
       'home_world_cups_before', 'away_is_host', 'away_goals_scored_last_4y',
       'away_goals_received_last_4y', 'away_wins_last_4y',
       'away_losses_last_4y', 'away_draws_last_4y',
       'away_fifa_rank_pre_tournament', 'away_fifa_points_pre_tournament',
       'away_world_cups_before', 'fifa_rank_diff', 'fifa_pt_diff',
       'home_squad_avg_age', 'away_squad_avg_age', 'squad_age_diff',
       'home_group_points', 'away_group_points', 'group_pts_diff',
       'home_rolling_avg_goals_scored', 'home_rolling_avg_goals_conceded',
       'away_rolling_avg_goals_scored', 'away_rolling_avg_goals_conceded',
       'rank_closeness', 'pts_closeness'],
      dtype='object')

In [ ]:
X_train = X
y_train = Y

model = RandomForestClassifier(n_estimators = 20, random_state = 723, class_weight = 'balanced')
model.fit(X_train, y_train)

probabilities = model.predict_proba(X_2026)
prob_df = pd.DataFrame(probabilities, columns=model.classes_)
results = pd.concat([matches_2026[['match_number', 'home_team_code', 'away_team_code']], prob_df], axis=1)
results.head()

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- away_world_cups_before
- home_world_cups_before


In [ ]:
results[['match_number','home_team_code','away_team_code','home_team_win','draw','away_team_win']].head(30)

,match_number,home_team_code,away_team_code,home_team_win,draw,away_team_win
0,1,MEX,RSA,0.46,0.24,0.30
1,2,KOR,CZE,0.50,0.19,0.31
2,3,CAN,BIH,0.33,0.26,0.41
3,4,USA,PAR,0.50,0.22,0.28
4,5,QAT,SUI,0.19,0.12,0.69
5,6,BRA,MAR,0.37,0.29,0.34
6,7,HAI,SCO,0.32,0.17,0.51
7,8,AUS,TUR,0.18,0.31,0.51
8,9,GER,CUR,0.34,0.18,0.48
9,10,NED,JPN,0.45,0.20,0.35


In [ ]:
kaggle_2026.tail()

,version,team,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,fifa_rank_pre_tournament,fifa_points_pre_tournament,squad_avg_age,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,team_code
43,WC-2026,Cape Verde,0,46,39,16,10,12,0,69,1366.13,29.2,0,0,0,0,0,0,CPV
44,WC-2026,Ghana,0,46,42,12,13,9,0,74,1346.31,25.1,4,2,2,1,0,0,GHA
45,WC-2026,Cura?o,0,55,39,12,10,10,0,82,1294.65,26.5,0,0,0,0,0,0,CUW
46,WC-2026,Haiti,0,71,37,16,8,8,0,83,1291.71,27.6,1,0,0,0,0,0,HAI
47,WC-2026,New Zealand,0,63,32,13,12,6,0,85,1281.57,27.8,2,0,0,0,0,0,NZL


In [ ]:
kaggle_2026[kaggle_2026['world_cup_participations_before'] == 0]

,version,team,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,fifa_rank_pre_tournament,fifa_points_pre_tournament,squad_avg_age,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,team_code
36,WC-2026,Uzbekistan,0,64,29,22,4,14,0,50,1465.34,25.6,0,0,0,0,0,0,UZB
41,WC-2026,Jordan,0,85,63,22,15,12,0,63,1391.45,28.5,0,0,0,0,0,0,JOR
43,WC-2026,Cape Verde,0,46,39,16,10,12,0,69,1366.13,29.2,0,0,0,0,0,0,CPV
45,WC-2026,Cura?o,0,55,39,12,10,10,0,82,1294.65,26.5,0,0,0,0,0,0,CUW
